# 11B — Regime-Gated Deep-Learning Ensemble

## Purpose
Script 11 showed that LSTM/GRU were **not uniformly superior**, but they became useful during some high-volatility periods, especially around 2022.

This notebook therefore tests one final DL hypothesis:

> Use Deep Learning only when the current market regime and already-resolved recent forecast errors justify it.

## Core rule

Script 09 remains the default forecast.

DL is allowed to contribute only when:

1. current volatility is above a historically estimated threshold; and
2. DL has lower recent resolved RMSE than Script 09.

No future observations are used in either decision.

## Inputs

- Script 04 Random Walk predictions
- Script 09 online-hybrid predictions
- Script 11 deep-learning predictions
- Script 02 development dataset for point-in-time volatility

## Anti-overfitting rule

A compact strategy grid is used. For each outer year, the strategy is selected using **earlier outer years only**. 2019 is a warm-up year and keeps Script 09.

After this notebook, the DL development branch should be frozen.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",200)
pd.set_option("display.width",230)
pd.set_option("display.float_format",lambda x:f"{x:,.6f}")

CURRENCIES=["USD","GBP","EUR","JPY"]
HORIZONS=[1,3,7]
YEARS=[2019,2020,2021,2022,2023]
MIN_HISTORY=20

# Small, prespecified strategy grid.
VOL_Q_GRID=[0.60,0.70,0.80]
ERR_WINDOWS=[20,40,80]
DL_WEIGHTS=[0.25,0.50,0.75]
ADVANTAGE_MARGINS=[0.00,0.005]

In [ ]:
def find_root():
    for root in [Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]:
        dev=root/"data"/"processed"/"02_development_model_dataset.csv"
        rw=root/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
        s09=root/"outputs"/"predictions"/"09_online_hybrid_predictions_2019_2023.csv"
        dl=root/"outputs"/"predictions"/"11_deep_learning_predictions_2019_2023.csv"
        if dev.exists() and rw.exists() and s09.exists() and dl.exists():
            return root.resolve()
    raise FileNotFoundError("Required Script 02/04/09/11 outputs not found.")

ROOT=find_root()
DEV_PATH=ROOT/"data"/"processed"/"02_development_model_dataset.csv"
PRED_DIR=ROOT/"outputs"/"predictions"
TABLE_DIR=ROOT/"outputs"/"tables"; TABLE_DIR.mkdir(parents=True,exist_ok=True)

RW_PATH=PRED_DIR/"04_random_walk_predictions_2019_2023.csv"
S09_PATH=PRED_DIR/"09_online_hybrid_predictions_2019_2023.csv"
DL_PATH=PRED_DIR/"11_deep_learning_predictions_2019_2023.csv"

print("Root:",ROOT)

In [ ]:
df=pd.read_csv(DEV_PATH,parse_dates=["Date"])
assert df["Date"].max()<pd.Timestamp("2024-01-01")

rw=pd.read_csv(RW_PATH,parse_dates=["Forecast_Origin","Target_Date"])
s09=pd.read_csv(S09_PATH,parse_dates=["Forecast_Origin","Target_Date"])
dl=pd.read_csv(DL_PATH,parse_dates=["Forecast_Origin","Target_Date"])

s09_col="Online_Hybrid_Prediction" if "Online_Hybrid_Prediction" in s09.columns else "Prediction"
dl_col="DL_Prediction" if "DL_Prediction" in dl.columns else "Final_DL_Hybrid_Prediction"

KEYS=["Currency","Horizon","Forecast_Origin"]
base=rw[KEYS+["Target_Date","Actual","RW_Prediction"]].copy()
base=base.merge(s09[KEYS+[s09_col]].rename(columns={s09_col:"SCRIPT09"}),on=KEYS,how="inner")
base=base.merge(dl[KEYS+[dl_col]].rename(columns={dl_col:"DL_RAW"}),on=KEYS,how="inner")
base["Validation_Year"]=base["Forecast_Origin"].dt.year

print("Matched prediction rows:",len(base))

## Point-in-time volatility regime

For each currency we calculate a rolling 20-day standard deviation of log returns using only information available at the forecast origin.

In [ ]:
# ============================================================
# BUILD 20-DAY VOLATILITY FEATURE FOR DL GATING — FIXED
# ============================================================

# Create rolling 20-day volatility for each currency
for cur in CURRENCIES:

    mid = pd.to_numeric(
        df[f"{cur}_mid"],
        errors="coerce"
    )

    ret = np.log(
        mid / mid.shift(1)
    )

    df[f"{cur}_VOL20_DL_GATE"] = (
        ret
        .rolling(
            20,
            min_periods=10
        )
        .std()
    )


# ------------------------------------------------------------
# Make sure Forecast_Origin is datetime
# ------------------------------------------------------------

base["Forecast_Origin"] = pd.to_datetime(
    base["Forecast_Origin"]
)

df["Date"] = pd.to_datetime(
    df["Date"]
)


# ------------------------------------------------------------
# Reset/create VOL20 safely
# ------------------------------------------------------------

base["VOL20"] = np.nan


# ------------------------------------------------------------
# Map each currency's volatility to its forecast dates
# ------------------------------------------------------------

for cur in CURRENCIES:

    vol_map = (
        df[
            [
                "Date",
                f"{cur}_VOL20_DL_GATE"
            ]
        ]
        .drop_duplicates(
            subset="Date"
        )
        .set_index("Date")[
            f"{cur}_VOL20_DL_GATE"
        ]
    )

    mask = (
        base["Currency"] == cur
    )

    base.loc[
        mask,
        "VOL20"
    ] = (
        base.loc[
            mask,
            "Forecast_Origin"
        ]
        .map(vol_map)
        .to_numpy()
    )


# ------------------------------------------------------------
# Diagnostic
# ------------------------------------------------------------

print(
    base[
        [
            "Currency",
            "Horizon",
            "Forecast_Origin",
            "VOL20"
        ]
    ].head(20)
)

print(
    "\nMissing VOL20:",
    base["VOL20"].isna().sum(),
    "/",
    len(base)
)

print("\nVOL20 summary by currency:")

print(
    base.groupby(
        "Currency"
    )["VOL20"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max"
        ]
    )
)

In [ ]:
def rmse(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    m=np.isfinite(y)&np.isfinite(p)
    return float(np.sqrt(np.mean((y[m]-p[m])**2))) if m.sum() else np.nan

def mae(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    m=np.isfinite(y)&np.isfinite(p)
    return float(np.mean(np.abs(y[m]-p[m]))) if m.sum() else np.nan

def skill(m,r): return float(1-m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan
def theils_u(m,r): return float(m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan

def dm_test(y,model,rw_pred,horizon=1):
    y=np.asarray(y,float); model=np.asarray(model,float); rw_pred=np.asarray(rw_pred,float)
    m=np.isfinite(y)&np.isfinite(model)&np.isfinite(rw_pred)
    y=y[m]; model=model[m]; rw_pred=rw_pred[m]
    n=len(y)
    if n<20: return np.nan,np.nan
    d=(y-rw_pred)**2-(y-model)**2
    lag=max(0,int(horizon)-1)
    lrv=np.var(d,ddof=1)
    for k in range(1,lag+1):
        if len(d)<=k: break
        cov=np.cov(d[k:],d[:-k],ddof=1)[0,1]
        lrv += 2*(1-k/(lag+1))*cov
    if not np.isfinite(lrv) or lrv<=0: return np.nan,np.nan
    stat=np.mean(d)/np.sqrt(lrv/n)
    p=2*student_t.sf(abs(stat),df=n-1)
    return float(stat),float(p)

## Sequential regime gate

For each forecast origin:

- historical volatility threshold is estimated from earlier origins only;
- recent model-performance comparison uses only forecasts whose target dates have already resolved;
- DL is activated only if both conditions pass.

In [ ]:
def simulate_strategy(task,vol_q,err_window,dl_weight,margin):
    task=task.sort_values("Forecast_Origin").reset_index(drop=True)
    rows=[]

    for i,row in task.iterrows():
        origin=row["Forecast_Origin"]

        # Historical rows available at current origin.
        past_origins=task[task["Forecast_Origin"]<origin]
        past_resolved=task[(task["Forecast_Origin"]<origin)&(task["Target_Date"]<=origin)]

        use_dl=False
        threshold=np.nan
        recent09=np.nan
        recentdl=np.nan

        if len(past_origins)>=MIN_HISTORY and len(past_resolved)>=MIN_HISTORY:
            hist_vol=past_origins["VOL20"].dropna()
            if len(hist_vol)>=MIN_HISTORY and np.isfinite(row["VOL20"]):
                threshold=float(hist_vol.quantile(vol_q))
                high_vol=float(row["VOL20"])>=threshold
            else:
                high_vol=False

            recent=past_resolved.tail(err_window)
            recent09=rmse(recent["Actual"],recent["SCRIPT09"])
            recentdl=rmse(recent["Actual"],recent["DL_RAW"])

            advantage=(recent09-recentdl)/recent09 if np.isfinite(recent09) and recent09>0 and np.isfinite(recentdl) else -np.inf
            use_dl=bool(high_vol and advantage>margin)

        if use_dl:
            pred=(1-dl_weight)*float(row["SCRIPT09"])+dl_weight*float(row["DL_RAW"])
        else:
            pred=float(row["SCRIPT09"])

        rows.append({
            "Currency":row["Currency"],"Horizon":row["Horizon"],
            "Forecast_Origin":origin,"Target_Date":row["Target_Date"],
            "Validation_Year":row["Validation_Year"],"Actual":row["Actual"],
            "RW_Prediction":row["RW_Prediction"],"SCRIPT09":row["SCRIPT09"],"DL_RAW":row["DL_RAW"],
            "Final_Prediction":pred,"DL_Activated":use_dl,"VOL20":row["VOL20"],
            "Vol_Threshold":threshold,"Recent_S09_RMSE":recent09,"Recent_DL_RMSE":recentdl
        })

    return pd.DataFrame(rows)

In [ ]:
STRATEGIES=[]
for q in VOL_Q_GRID:
    for w in ERR_WINDOWS:
        for a in DL_WEIGHTS:
            for m in ADVANTAGE_MARGINS:
                STRATEGIES.append({"vol_q":q,"err_window":w,"dl_weight":a,"margin":m})
print("Candidate strategies:",len(STRATEGIES))

In [ ]:
all_sims=[]
for cur in CURRENCIES:
    for h in HORIZONS:
        task=base[(base["Currency"]==cur)&(base["Horizon"]==h)].copy()
        print(f"Simulating {cur} H{h}...",flush=True)
        for sid,s in enumerate(STRATEGIES):
            sim=simulate_strategy(task,**s)
            sim["Strategy_ID"]=sid
            all_sims.append(sim)

sims=pd.concat(all_sims,ignore_index=True)
print("Simulation rows:",len(sims))

## Year-by-year strategy selection

2019 keeps Script 09.

For 2020 onward, the regime-gate strategy is selected using only earlier outer years.

In [ ]:
final_frames=[]; sel_rows=[]

for cur in CURRENCIES:
    for h in HORIZONS:
        x=sims[(sims["Currency"]==cur)&(sims["Horizon"]==h)].copy()

        for year in YEARS:
            if year==2019:
                # Warm-up: exact Script09.
                y=x[(x["Validation_Year"]==year)&(x["Strategy_ID"]==0)].copy()
                y["Final_Prediction"]=y["SCRIPT09"]
                chosen=-1
                hist_rmse=np.nan
                params={"vol_q":np.nan,"err_window":np.nan,"dl_weight":0.0,"margin":np.nan}
            else:
                prior=x[x["Validation_Year"]<year]
                scores=[]
                for sid,g in prior.groupby("Strategy_ID"):
                    scores.append((sid,rmse(g["Actual"],g["Final_Prediction"])))
                scores=sorted(scores,key=lambda z:(z[1],z[0]))
                chosen=int(scores[0][0]); hist_rmse=float(scores[0][1])
                params=STRATEGIES[chosen]
                y=x[(x["Validation_Year"]==year)&(x["Strategy_ID"]==chosen)].copy()

            y["Selected_Strategy_ID"]=chosen
            final_frames.append(y)
            sel_rows.append({"Currency":cur,"Horizon":h,"Outer_Year":year,
                             "Selected_Strategy_ID":chosen,"Historical_RMSE":hist_rmse,**params})

final_predictions=pd.concat(final_frames,ignore_index=True)
selections=pd.DataFrame(sel_rows)
display(selections)

In [ ]:
pooled=[]
for (cur,h),g in final_predictions.groupby(["Currency","Horizon"]):
    rr=rmse(g["Actual"],g["RW_Prediction"])
    r09=rmse(g["Actual"],g["SCRIPT09"])
    rf=rmse(g["Actual"],g["Final_Prediction"])
    _,p=dm_test(g["Actual"],g["Final_Prediction"],g["RW_Prediction"],h)
    pooled.append({"Currency":cur,"Horizon":h,"Observations":len(g),
                   "RW_RMSE":rr,"Script09_RMSE":r09,"Regime_DL_RMSE":rf,
                   "Script09_Skill":skill(r09,rr),"Regime_DL_Skill":skill(rf,rr),
                   "Theils_U":theils_u(rf,rr),"DM_pvalue_vs_RW":p,
                   "Beats_RW":rf<rr,"DL_Activation_Rate":float(g["DL_Activated"].mean())})

pooled_results=pd.DataFrame(pooled).sort_values(["Currency","Horizon"])
display(pooled_results)
print("Tasks beating RW:",int(pooled_results["Beats_RW"].sum()),"/",len(pooled_results))

In [ ]:
yearly=[]
for (cur,h,year),g in final_predictions.groupby(["Currency","Horizon","Validation_Year"]):
    rr=rmse(g["Actual"],g["RW_Prediction"]); rf=rmse(g["Actual"],g["Final_Prediction"])
    _,p=dm_test(g["Actual"],g["Final_Prediction"],g["RW_Prediction"],h)
    yearly.append({"Currency":cur,"Horizon":h,"Validation_Year":year,"Observations":len(g),
                   "RMSE":rf,"MAE":mae(g["Actual"],g["Final_Prediction"]),
                   "Forecast_Skill_vs_RW":skill(rf,rr),"Theils_U":theils_u(rf,rr),
                   "DM_pvalue_vs_RW":p,"DL_Activation_Rate":float(g["DL_Activated"].mean())})

yearly_results=pd.DataFrame(yearly).sort_values(["Currency","Horizon","Validation_Year"])
display(yearly_results)

In [ ]:
final_predictions.to_csv(PRED_DIR/"11B_regime_gated_DL_predictions_2019_2023.csv",index=False)
pooled_results.to_csv(TABLE_DIR/"11B_regime_gated_DL_pooled_results.csv",index=False)
yearly_results.to_csv(TABLE_DIR/"11B_regime_gated_DL_yearly_results.csv",index=False)
selections.to_csv(TABLE_DIR/"11B_regime_gated_DL_strategy_selections.csv",index=False)

quality_gate=pd.DataFrame([
    {"Area":"Holdout isolation","Status":"PASS","Evidence":"2024–2025 not loaded"},
    {"Area":"Regime variable","Status":"PASS","Evidence":"20-day volatility uses current/past prices only"},
    {"Area":"Performance gate","Status":"PASS","Evidence":"Only resolved forecast errors used"},
    {"Area":"Common rule","Status":"PASS","Evidence":"Same strategy grid for all currencies/horizons"},
    {"Area":"Outer-year selection","Status":"PASS","Evidence":"Strategy selected from earlier outer years only"},
    {"Area":"Script09 safeguard","Status":"PASS","Evidence":"Script09 remains default when gate fails"}
])
display(quality_gate)
quality_gate.to_csv(TABLE_DIR/"11B_regime_gated_DL_quality_gate.csv",index=False)

print("\nSCRIPT 11B COMPLETE")
print("Tasks beating RW:",int(pooled_results["Beats_RW"].sum()),"/",len(pooled_results))
print("Freeze DL branch after this experiment.")
print("2024–2025 remains untouched.")